# 01. MCP (Model Context Protocol) with LangChain

> **주제**: 에이전트가 *외부 도구/데이터*에 표준 방식으로 접근하기 — `langchain-mcp-adapters`
>
> **원문**: https://docs.langchain.com/oss/python/langchain/mcp

---

## 이 노트북에서 배우는 것

1. MCP 가 무엇이고 왜 필요한가 (USB-C 비유)
2. `MultiServerMCPClient` 로 여러 MCP 서버의 도구를 한 번에 불러오기
3. `stdio` vs `http` 전송 방식
4. `FastMCP` 로 나만의 MCP 서버 만들기
5. 상태 유지 세션, 리소스/프롬프트, 인터셉터, 콜백, Elicitation

각 셀은 **개념 설명 → 실행 코드** 순서로 구성되어 있습니다.

## 1. MCP 한눈에 보기

**MCP(Model Context Protocol)** 는 LLM 애플리케이션이 외부 도구·데이터 소스에 연결하는 방식을 표준화한 오픈 프로토콜입니다.

> 흔히 **"AI 애플리케이션을 위한 USB-C"** 라고 비유합니다. 도구마다 제각각인 연결 방식을 하나의 규격으로 통일하는 것이죠.

LangChain 에서는 `langchain-mcp-adapters` 패키지가 MCP 서버가 노출하는 **tools / resources / prompts** 를 LangChain 의 `BaseTool` 객체로 변환해 줍니다. 그래서 한 번 변환하면 일반 LangChain 도구처럼 `create_agent` 에 바로 넘길 수 있습니다.

```
[LangChain Agent] --(MCP adapters)--> [MCP Server 1: math]
                                  \-> [MCP Server 2: weather]
```

In [14]:
# 설치 (uv)
!uv pip install -q python-dotenv langchain-mcp-adapters langchain "langchain[anthropic]"
# uv 프로젝트라면:  uv add langchain-mcp-adapters

# 직접 MCP 서버를 만들어 볼 때 추가로:
!uv pip install -q fastmcp

from dotenv import load_dotenv
load_dotenv("/home/shn413.jung/work/toy/study/langchain-docs/deep-agents/week3-protocols-sehun/.env", override=True)

True

## 2. 먼저 MCP 서버 두 개를 만든다 (FastMCP)

클라이언트를 실험하려면 붙을 서버가 필요합니다. `FastMCP` 로 아주 작은 서버 두 개를 파일로 떨궈 둡니다.

- **math 서버**: `stdio` 전송 — 클라이언트가 서브프로세스로 실행
- **weather 서버**: `streamable-http` 전송 — 별도 포트에서 동작

`@mcp.tool()` 로 데코레이트된 함수가 그대로 도구가 되고, **docstring 이 도구 설명** 으로 쓰여 LLM 이 언제 호출할지 판단합니다.

In [15]:
# math_server.py 작성 (stdio)
math_server = '''
from fastmcp import FastMCP

mcp = FastMCP("Math")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("math_server.py", "w") as f:
    f.write(math_server)
print("math_server.py 생성 완료")

math_server.py 생성 완료


In [16]:
# weather_server.py 작성 (streamable-http)
weather_server = '''
from fastmcp import FastMCP

mcp = FastMCP("Weather")

@mcp.tool()
async def get_weather(location: str) -> str:
    """Get weather for location."""
    return "It\'s always sunny in New York"

if __name__ == "__main__":
    mcp.run(transport="streamable-http", host="127.0.0.1", port=8765)
'''

with open("weather_server.py", "w") as f:
    f.write(weather_server)
print("weather_server.py 생성 완료")

# weather 서버는 HTTP 이므로 백그라운드로 띄워 둔다 (터미널에서 실행해도 됨):
#   python weather_server.py   ->  http://localhost:8000/mcp

weather_server.py 생성 완료


## 3. `MultiServerMCPClient` — 여러 서버를 한 번에

핵심 클래스입니다. 서버 이름을 키로 갖는 딕셔너리로 **여러 MCP 서버**를 동시에 등록하고, `get_tools()` 한 번으로 모든 도구를 모읍니다.

전송 방식별 설정 항목:

| 전송 | 필수 키 | 설명 |
|------|---------|------|
| `stdio` | `command`, `args` | 로컬 서브프로세스로 서버 실행 |
| `http`  | `url` (옵션 `headers`) | 원격/로컬 HTTP 서버에 연결 |

> ⚠️ `MultiServerMCPClient` 는 **기본적으로 stateless** 입니다. 도구를 호출할 때마다 세션을 새로 열고 → 실행 → 정리합니다. 호출 간 상태가 필요하면 7장의 `client.session()` 을 쓰세요.

In [17]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

client = MultiServerMCPClient(
    {
        "math": {
            "transport": "stdio",
            "command": "python",
            "args": ["/home/shn413.jung/work/toy/study/langchain-docs/deep-agents/week3-protocols-sehun/math_server.py"],  # 위에서 만든 파일 (절대경로 권장)
        },
        "weather": {
            "transport": "http",
            "url": "http://localhost:8765/mcp",
        },
    }
)

# 모든 서버의 도구를 LangChain 도구로 로드
tools = await client.get_tools()
print("로드된 도구:", [t.name for t in tools])

로드된 도구: ['add', 'multiply', 'get_weather']


In [31]:
# 에이전트에 도구를 그대로 연결한다
# (OpenAI 모델 사용 — provider:model 형식. 키는 OPENAI_API_KEY 환경변수)
agent = create_agent("openai:gpt-4o-mini", tools) # 기존: "claude-sonnet-4-6"

# result = await agent.ainvoke(
#     {"messages": [{"role": "user", "content": "what's (3 + 5) x 12?"}]}
# )
result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "what's the weather in New York?"}]}
)
print(result["messages"][-1].content)

I'm unable to provide real-time weather information. However, you can easily find the current weather in New York by checking a reliable weather website or using a weather app.


## 4. HTTP 전송 + 인증 헤더

원격 MCP 서버는 보통 토큰 인증을 요구합니다. `headers` 로 `Authorization` 을 넘깁니다.

In [20]:
client = MultiServerMCPClient(
    {
        "weather": {
            "transport": "http",
            "url": "http://localhost:8765/mcp",
            "headers": {"Authorization": "Bearer 123123123"},
        }
    }
)

## 5. 상태 유지 세션 (`client.session()`)

기본 stateless 동작과 달리, **하나의 연결을 열어 두고** 여러 도구 호출에서 컨텍스트를 공유하고 싶을 때 사용합니다. (예: 로그인 세션, 커서 위치 등)

In [21]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools

client = MultiServerMCPClient({
    "math": {
        "transport": "stdio",
        "command": "python",
        "args": ["math_server.py"],
    }
})

async with client.session("math") as session:
    tools = await load_mcp_tools(session)          # 이 세션에 묶인 도구
    agent = create_agent("openai:gpt-4o-mini", tools)
    # 이 블록 안에서의 호출은 같은 세션을 공유한다

## 6. 구조화된 결과 / 리소스 / 프롬프트

MCP 도구는 텍스트뿐 아니라 **구조화된 콘텐츠**를 반환할 수 있습니다. 결과는 `ToolMessage.artifact` 에 담깁니다.

또한 서버는 도구 외에도 **resources**(읽을 수 있는 데이터 blob)와 **prompts**(재사용 프롬프트 템플릿)를 노출할 수 있습니다.

In [29]:
from langchain.messages import ToolMessage

result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "what's (3 + 5) x 12?"}]}
)

for message in result["messages"]:
    if isinstance(message, ToolMessage) and message.artifact:
        structured_content = message.artifact["structured_content"]
        print(structured_content)

ClosedResourceError: 

In [9]:
# 리소스 로드: 서버가 노출하는 파일/데이터 blob
blobs = await client.get_resources("server_name")
# 특정 URI 만 선택할 수도 있다
blobs = await client.get_resources(
    "server_name", uris=["file:///path/to/file.txt"]
)

for blob in blobs:
    print(f"URI: {blob.metadata['uri']}")
    print(blob.as_string())

ValueError: Couldn't find a server with name 'server_name', expected one of '['math']'

In [ ]:
# 프롬프트 로드: 서버가 제공하는 재사용 프롬프트
messages = await client.get_prompt("server_name", "summarize")

# 인자가 있는 프롬프트
messages = await client.get_prompt(
    "server_name",
    "code_review",
    arguments={"language": "python", "focus": "security"},
)

## 7. 도구 인터셉터 — 호출 전후 가로채기

인터셉터는 미들웨어처럼 동작합니다. `handler` 를 호출하기 전/후로 로깅, 인자 변형, 런타임 컨텍스트 주입 등을 할 수 있습니다.

In [13]:
from langchain_mcp_adapters.interceptors import MCPToolCallRequest

# (1) 로깅 인터셉터
async def logging_interceptor(request: MCPToolCallRequest, handler):
    print(f"Calling: {request.name} with {request.args}")
    result = await handler(request)
    return result

client = MultiServerMCPClient({}, tool_interceptors=[logging_interceptor])

In [14]:
# (2) 인자 변형: 모든 인자를 2배로
async def double_args_interceptor(request: MCPToolCallRequest, handler):
    modified_args = {k: v * 2 for k, v in request.args.items()}
    modified_request = request.override(args=modified_args)
    return await handler(modified_request)

# (3) 런타임 컨텍스트 주입: user_id 자동 첨부
async def inject_user_context(request: MCPToolCallRequest, handler):
    runtime = request.runtime
    user_id = runtime.context.user_id
    modified_request = request.override(
        args={**request.args, "user_id": user_id}
    )
    return await handler(modified_request)

## 8. 진행률/로그 콜백 & Elicitation

- **콜백**: 장시간 도구의 진행률(`on_progress`)이나 로그를 수신
- **Elicitation**: 도구 실행 중 서버가 *사용자에게 추가 정보*를 요청하는 양방향 흐름

In [ ]:
from langchain_mcp_adapters.callbacks import Callbacks, CallbackContext

async def on_progress(progress, total, message, context):
    percent = (progress / total * 100) if total else progress
    print(f"[{context.server_name}] Progress: {percent:.1f}%")

client = MultiServerMCPClient({}, callbacks=Callbacks(on_progress=on_progress))

In [ ]:
# Elicitation — 서버측 (도구가 사용자 입력을 요청)
from pydantic import BaseModel
from mcp.server.fastmcp import Context, FastMCP

class UserDetails(BaseModel):
    email: str
    age: int

# @server.tool()
async def create_profile(name: str, ctx: Context) -> str:
    result = await ctx.elicit(
        message=f"Provide details for {name}'s profile:",
        schema=UserDetails,
    )
    ...

# Elicitation — 클라이언트측 (요청에 응답)
from mcp.types import ElicitResult

async def on_elicitation(mcp_context, params, context):
    return ElicitResult(
        action="accept",
        content={"email": "user@example.com", "age": 25},
    )

client = MultiServerMCPClient(
    {}, callbacks=Callbacks(on_elicitation=on_elicitation)
)

## 정리 & 연습 문제

**핵심 요약**
- MCP = 도구/데이터 연결의 표준 규격. `langchain-mcp-adapters` 가 MCP → LangChain 도구로 변환
- `MultiServerMCPClient({이름: {transport, ...}})` → `get_tools()` → `create_agent`
- 전송: `stdio`(로컬 프로세스) / `http`(원격)
- 기본 stateless, 필요하면 `client.session()`
- 고급: 구조화 결과(`artifact`), resources/prompts, 인터셉터, 콜백, elicitation

**연습**
1. `math_server.py` 에 `subtract` 도구를 추가하고 에이전트로 `(20 - 8) x 3` 을 계산시켜 보세요.
2. `logging_interceptor` 를 실제 클라이언트에 연결해 도구 호출 로그를 출력해 보세요.
3. weather 서버를 `http` 로 띄우고 "What's the weather in New York?" 을 물어보세요.